### Class Weights 

Add class weights to the baseline models and report results. 

Don't include the MLP Classifier because the sklearn implementation doesn't offer class weight support. 

In [1]:
import sys
from pathlib import Path

# cwd can be repo root, src/, or src/feature_selection/ — walk up until src/stroke_data.py exists
_here = Path().resolve()
REPO_ROOT = _here
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "src" / "stroke_data.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError("Could not find src/stroke_data.py (open this project from the repo folder).")

sys.path.insert(0, str(REPO_ROOT / "src"))

In [2]:
import sklearn
import scipy
import numpy as np
from stroke_data import get_stroke_data_for_cv, get_stroke_data

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [3]:
X_train, X_test, y_train, y_test = get_stroke_data_for_cv("data/knn-standardize-distance.csv")

In [5]:
results = {}

#### Logistic Regression

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [6]:
lr = LogisticRegression(C=0.001, class_weight='balanced', solver='liblinear')

lr.fit(X=X_train, y=y_train)

lr_train_preds = lr.predict(X_train)
lr_preds = lr.predict(X_test)

results['lr'] = {
    "train": {"accuracy": accuracy_score(y_train, lr_train_preds),
                "f1": f1_score(y_train, lr_train_preds),
                "precision": precision_score(y_train, lr_train_preds),
                "recall": recall_score(y_train, lr_train_preds)},
    "test": {"accuracy": accuracy_score(y_test, lr_preds),
                "f1": f1_score(y_test, lr_preds),
                "precision": precision_score(y_test, lr_preds),
                "recall": recall_score(y_test, lr_preds)},
}

### SVM 

https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

In [7]:
svm = SVC(C=10.0, class_weight='balanced', gamma=1, kernel='rbf')

svm.fit(X=X_train, y=y_train)

svm_train_preds = svm.predict(X_train)
svm_preds = svm.predict(X_test)

results['svm'] = {
    "train": {"accuracy": accuracy_score(y_train, svm_train_preds),
                "f1": f1_score(y_train, svm_train_preds),
                "precision": precision_score(y_train, svm_train_preds),
                "recall": recall_score(y_train, svm_train_preds)},
    "test": {"accuracy": accuracy_score(y_test, svm_preds),
                "f1": f1_score(y_test, svm_preds),
                "precision": precision_score(y_test, svm_preds),
                "recall": recall_score(y_test, svm_preds)},
}

### Random Forests 
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [8]:
rf = RandomForestClassifier(bootstrap=False, class_weight='balanced_subsample', max_depth=None, max_features=None, max_leaf_nodes=None, n_estimators=15)

rf.fit(X=X_train, y=y_train)

rf_train_preds = rf.predict(X_train)
rf_preds = rf.predict(X_test)

results['rf'] = {
    "train": {"accuracy": accuracy_score(y_train, rf_train_preds),
                "f1": f1_score(y_train, rf_train_preds),
                "precision": precision_score(y_train, rf_train_preds),
                "recall": recall_score(y_train, rf_train_preds)},
    "test": {"accuracy": accuracy_score(y_test, rf_preds),
                "f1": f1_score(y_test, rf_preds),
                "precision": precision_score(y_test, rf_preds),
                "recall": recall_score(y_test, rf_preds)},
}

### XGBoost

https://xgboost.readthedocs.io/en/latest/parameter.html

https://xgboost.readthedocs.io/en/latest/python/sklearn_estimator.html

In [9]:
xgb = XGBClassifier(eta=1, gamma=2, reg_lambda=0.5, max_depth=6, objective='binary:logistic', subsample=0.1, scale_pos_weight=19)

xgb.fit(X=X_train, y=y_train)

xgb_train_preds = xgb.predict(X_train)
xgb_preds = xgb.predict(X_test)

results['xgb'] = {
    "train": {"accuracy": accuracy_score(y_train, xgb_train_preds),
                "f1": f1_score(y_train, xgb_train_preds),
                "precision": precision_score(y_train, xgb_train_preds),
                "recall": recall_score(y_train, xgb_train_preds)},
    "test": {"accuracy": accuracy_score(y_test, xgb_preds),
                "f1": f1_score(y_test, xgb_preds),
                "precision": precision_score(y_test, xgb_preds),
                "recall": recall_score(y_test, xgb_preds)},
}

### Naive Bayes

https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB

In [10]:
gnb = GaussianNB(priors=[0.5, 0.5], var_smoothing=1e-11)

gnb.fit(X=X_train, y=y_train)

gnb_preds = gnb.predict(X_test)
gnb_train_preds = gnb.predict(X_train)

results['gnb'] = {
    "train": {"accuracy": accuracy_score(y_train, gnb_train_preds),
                "f1": f1_score(y_train, gnb_train_preds),
                "precision": precision_score(y_train, gnb_train_preds),
                "recall": recall_score(y_train, gnb_train_preds)},
    "test": {"accuracy": accuracy_score(y_test, gnb_preds),
                "f1": f1_score(y_test, gnb_preds),
                "precision": precision_score(y_test, gnb_preds),
                "recall": recall_score(y_test, gnb_preds)},
}

### KNN

https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

In [12]:
knn = KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski', n_neighbors=1, weights='uniform')

knn.fit(X=X_train, y=y_train)

knn_preds = knn.predict(X_test)
knn_train_preds = knn.predict(X_train)


results['knn'] = {
    "train": {"accuracy": accuracy_score(y_train, knn_train_preds),
                "f1": f1_score(y_train, knn_train_preds),
                "precision": precision_score(y_train, knn_train_preds),
                "recall": recall_score(y_train, knn_train_preds)},
    "test": {"accuracy": accuracy_score(y_test, knn_preds),
                "f1": f1_score(y_test, knn_preds),
                "precision": precision_score(y_test, knn_preds),
                "recall": recall_score(y_test, knn_preds)},
}

## Results

In [13]:
results

{'lr': {'train': {'accuracy': 0.62793542074364,
   'f1': 0.19052687599787121,
   'precision': 0.10654761904761904,
   'recall': 0.8994974874371859},
  'test': {'accuracy': 0.6477495107632094,
   'f1': 0.18552036199095023,
   'precision': 0.10459183673469388,
   'recall': 0.82}},
 'svm': {'train': {'accuracy': 0.9770058708414873,
   'f1': 0.8089430894308943,
   'precision': 0.6791808873720137,
   'recall': 1.0},
  'test': {'accuracy': 0.8943248532289628,
   'f1': 0.03571428571428571,
   'precision': 0.03225806451612903,
   'recall': 0.04}},
 'rf': {'train': {'accuracy': 1.0, 'f1': 1.0, 'precision': 1.0, 'recall': 1.0},
  'test': {'accuracy': 0.9246575342465754,
   'f1': 0.1348314606741573,
   'precision': 0.15384615384615385,
   'recall': 0.12}},
 'xgb': {'train': {'accuracy': 0.5105185909980431,
   'f1': 0.1400945423291792,
   'precision': 0.07659774436090226,
   'recall': 0.8190954773869347},
  'test': {'accuracy': 0.4931506849315068,
   'f1': 0.12794612794612795,
   'precision': 0.06